# Acne Detection Demo using YOLOv5

This notebook demonstrates how to use YOLOv5 for acne detection. We'll cover:
1. Loading the YOLOv5s model
2. Processing input images
3. Running inference
4. Visualizing detection results

## 1. Import Required Libraries

First, let's import all the necessary libraries for our acne detection task.

In [1]:
# Standard imports
import os
import sys
import warnings

# Deep learning imports
import torch
from ultralytics import YOLO

# Image processing
import cv2
import numpy as np
from PIL import Image

# Visualization
import matplotlib.pyplot as plt

# Suppress warnings
warnings.filterwarnings('ignore')

## 2. Load YOLOv5s Model

Now we'll load the YOLOv5s model using the weights file located at `yolov5s.pt`.

In [2]:
from pathlib import Path
import os
import torch
import pathlib

# Define model path (using os.path for Windows compatibility)
weights_path = os.path.abspath('yolo_acne_detection/acne_localization4/weights/best.pt')
weights_path = weights_path.replace('\\', '/')  # Convert to forward slashes

print(f"Loading model from: {weights_path}")

try:
    # Apply PosixPath fix for Windows
    posix_backup = pathlib.PosixPath
    pathlib.PosixPath = pathlib.WindowsPath
    
    try:
        # Load the model using torch.hub
        model = torch.hub.load('ultralytics/yolov5', 'custom', 
                             path=weights_path,
                             force_reload=True,
                             trust_repo=True)
        
        # Set model parameters
        model.conf = 0.25  # Confidence threshold
        model.iou = 0.45   # NMS IoU threshold
        
        # Ensure model is in evaluation mode
        model.eval()
        print("Model loaded successfully!")
        
    finally:
        # Restore original PosixPath
        pathlib.PosixPath = posix_backup

except Exception as e:
    print(f"Error loading model: {str(e)}")
    raise

Loading model from: c:/Users/rohan/Desktop/AcneDetection/yolo_acne_detection/acne_localization4/weights/best.pt
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\rohan/.cache\torch\hub\master.zip
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\rohan/.cache\torch\hub\master.zip


YOLOv5  2025-6-30 Python-3.10.0 torch-2.7.1+cpu CPU

Fusing layers... 
Fusing layers... 
Model summary: 157 layers, 7026307 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
Model summary: 157 layers, 7026307 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Model loaded successfully!


## 3. Load and Preprocess Input Image

Let's load a test image from our dataset and prepare it for inference.

In [3]:
# Load a test image from the test dataset
test_image_path = 'C://Users//rohan//Desktop//AcneDetection//test.jpg'

# Load and display the image
img = Image.open(test_image_path)
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()

## 4. Run Inference with YOLOv5s

Now we'll run the model on our test image to detect acne lesions.

In [4]:
# Run inference
import cv2
import torch
import numpy as np

try:
    # Make prediction
    results = model(test_image_path)
    
    # Print detection summary
    print("\nDetection Results:")
    print("=" * 50)
    print(f"Detected {len(results.pred[0])} objects")
    
    # Print detailed results
    for *box, conf, cls in results.pred[0]:
        class_name = results.names[int(cls)]
        print(f"\nClass: {class_name}")
        print(f"Confidence: {conf:.2f}")
        print(f"Bounding Box: {[float(x) for x in box]}")
        
except Exception as e:
    print(f"Error during inference: {str(e)}")
    
    # Try alternative approach with manual preprocessing
    try:
        print("\nAttempting alternative inference method...")
        # Read and preprocess image
        img = cv2.imread(test_image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Make prediction
        results = model(img)
        print("Alternative inference successful!")
        
        # Print detection summary
        print("\nDetection Results:")
        print("=" * 50)
        print(f"Detected {len(results.pred[0])} objects")
        
        # Print detailed results
        for *box, conf, cls in results.pred[0]:
            class_name = results.names[int(cls)]
            print(f"\nClass: {class_name}")
            print(f"Confidence: {conf:.2f}")
            print(f"Bounding Box: {[float(x) for x in box]}")
            
    except Exception as e2:
        print(f"Alternative inference failed: {str(e2)}")
        raise


Detection Results:
Detected 1 objects

Class: papules
Confidence: 0.41
Bounding Box: [969.9938354492188, 1027.980712890625, 1149.5489501953125, 1179.27978515625]


## 5. Visualize Detection Results

Finally, let's visualize the detection results by drawing bounding boxes on the image.

In [5]:
# Visualize results
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

def plot_results(image_path, results, save_dir='predictions'):
    # Create output directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Read image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Create a copy for drawing
    img_draw = img.copy()
    
    # Draw each detection
    for *box, conf, cls in results.pred[0]:
        # Get box coordinates
        x1, y1, x2, y2 = map(int, box)
        
        # Get class name
        class_name = results.names[int(cls)]
        
        # Draw rectangle
        cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Add label
        label = f"{class_name}: {conf:.2f}"
        cv2.putText(img_draw, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Display results
    plt.figure(figsize=(12, 8))
    plt.imshow(img_draw)
    plt.axis('off')
    plt.title('Acne Detection Results')
    plt.show()
    
    # Save the image
    output_filename = os.path.join(save_dir, 'prediction_' + os.path.basename(image_path))
    cv2.imwrite(output_filename, cv2.cvtColor(img_draw, cv2.COLOR_RGB2BGR))
    print(f"Saved prediction image to: {output_filename}")

try:
    # Visualize detections
    plot_results(test_image_path, results)
except Exception as e:
    print(f"Error visualizing results: {str(e)}")
    # Try alternative visualization using model's built-in plotting
    try:
        # Create predictions directory
        os.makedirs('predictions', exist_ok=True)
        
        # Use model's built-in rendering
        rendered_img = results.render()[0]
        
        # Display
        plt.figure(figsize=(12, 8))
        plt.imshow(rendered_img)
        plt.axis('off')
        plt.title('Acne Detection Results (Alternative Method)')
        plt.show()
        
        # Save using alternative method
        output_filename = os.path.join('predictions', 'prediction_alt_' + os.path.basename(test_image_path))
        cv2.imwrite(output_filename, rendered_img)
        print(f"Saved prediction image to: {output_filename}")
        
    except Exception as e2:
        print(f"Error with alternative visualization: {str(e2)}")
        raise

Saved prediction image to: predictions\prediction_test.jpg
